In [ ]:
!pip install qiskit==1.4.0 qiskit-aer numpy

In [2]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit_aer.primitives import SamplerV2 as Sampler

In [3]:
# Problem instance

weights = [2, 3, 4, 5]
values = [3, 4, 5, 6]
capacity = 8
num_items = len(weights)
population_size = 10  # Number of measurements per generation
num_generations = 5

In [ ]:
# Initialize quantum circuit with parameters (initially all zeros)

def create_quantum_circuit(parameters):
    qc = QuantumCircuit(num_items, num_items)
    for i in range(num_items):
        qc.h(i)                 # Initial superposition
        qc.ry(parameters[i], i) # Parameterized rotation to bias probabilities
    qc.measure(range(num_items), range(num_items))
    
    return qc

In [ ]:
# Fitness evaluation function

def evaluate_fitness(individual):
    total_weight, total_value = 0, 0
    for i in range(num_items):
        if individual[i] == '1':
            total_weight += weights[i]
            total_value += values[i]

    # Penalize overweight solutions
    if total_weight > capacity:
        return 0
    
    return total_value

In [6]:
# Selection (Roulette Wheel - simplified)

def selection(population_fitness, population):
    total_fitness = sum(population_fitness)
    if total_fitness == 0:
        return np.random.choice(population, size=len(population), replace=True)
    
    probabilities = [f / total_fitness for f in population_fitness]
    indices = np.random.choice(len(population), size=len(population), p=probabilities, replace=True)
    
    return [population[i] for i in indices]

In [7]:
# Crossover (Single-point - simplified)

def crossover(parents):
    offspring = []
    for i in range(0, len(parents), 2):
        parent1 = parents[i]
        parent2 = parents[(i + 1) % len(parents)]
        crossover_point = np.random.randint(1, num_items)
        offspring.append(parent1[:crossover_point] + parent2[crossover_point:])
        offspring.append(parent2[:crossover_point] + parent1[crossover_point:])
    
    return offspring

In [8]:
# Mutation (Bit-flip - simplified)

def mutation(population, mutation_rate=0.1):
    mutated_population = []
    for individual in population:
        mutated_individual = list(individual)
        for i in range(num_items):
            if np.random.rand() < mutation_rate:
                mutated_individual[i] = '1' if mutated_individual[i] == '0' else '0'
        mutated_population.append("".join(mutated_individual))
    
    return mutated_population

In [9]:
# Main GQA loop

simulator = Sampler()
parameters = np.zeros(num_items) # Initialize parameters to create equal superposition

for generation in range(num_generations):
    # 1. Create and run quantum circuit
    qc = create_quantum_circuit(parameters)
    job = simulator.run([qc], shots=1000)
    result = job.result()
    counts = result[0].data.meas.get_counts()

    # 2. Get classical population from measurements
    population = []
    for individual in counts:
        population.extend([individual] * counts[individual])

    if not population:
        print(f"Generation {generation}: No valid solutions found.")
        break

    # 3. Evaluate fitness of the population
    population_fitness = [evaluate_fitness(individual) for individual in population]

    # Print best fitness in the current generation
    best_fitness = max(population_fitness) if population_fitness else 0
    best_individual = population[np.argmax(population_fitness)] if population_fitness else "N/A"
    print(f"Generation {generation}: Best Fitness = {best_fitness}, Best Individual = {best_individual}")

    # 4. Selection
    parents = selection(population_fitness, population)

    # 5. Crossover
    offspring = crossover(parents)

    # 6. Mutation
    mutated_offspring = mutation(offspring)

    # 7. Update parameters of the quantum circuit (simplified - based on best individual)
    best_solution = best_individual
    new_parameters = np.zeros(num_items)
    for i in range(num_items):
        if best_solution[i] == '1':
            new_parameters[i] = np.pi / 3  # Bias towards '1'
        else:
            new_parameters[i] = 0          # Bias towards '0'

    parameters = new_parameters

# Final run to get the best solution

final_qc = create_quantum_circuit(parameters)
final_job = simulator.run([final_qc], shots=1000)
final_result = final_job.result()
final_counts = final_result[0].data.meas.get_counts()

print("\nFinal Results:")
print(final_counts)

best_final_solution = max(final_counts, key=final_counts.get)
best_final_fitness = evaluate_fitness(best_final_solution)

print(f"Best Final Solution: {best_final_solution}, Fitness: {best_final_fitness}")

AttributeError: 'DataBin' object has no attribute 'meas'